<span style="font-family: 'Time New Roman '; font-size: 20px;">
This code provides three tools to the LLM:
<br></br>
<table>
<tr><td>get_os_info()</td><td>this returns information about the current operating system.</td></tr>
<tr><td>search_web(query)</td><td>this provides a brief summary of some queries.</td></tr>
<tr><td>get_share_price(ticker_symbol)</td><td>this provides the current share price for a valid ticker symbol.</td></tr>
</table>

Multiple calls to these tools can be performed prior to returning the outcome to the user.
</span> 

In [ ]:
import os
import platform
import sys

<span style="font-family: 'Time New Roman '; font-size: 20px;">
The following cell defines a Tool called get_os_info(), <br> it takes no arguments and returns details of the system on which it is run.
</span> 

In [ ]:
def get_os_info():
    print("Called the os_info function.")
    try:
        info = {
            "OS Name": os.name,  # 'posix', 'nt', etc.
            "Platform System": platform.system(),  # e.g., 'Windows', 'Linux', 'Darwin'
            "Platform Release": platform.release(),  # OS release version
            "Platform Version": platform.version(),  # Detailed version/build
            "Architecture": platform.machine(),  # e.g., 'x86_64', 'AMD64', 'arm64'
            "Processor": platform.processor(),  # CPU info (may be empty on some OS)
            "Python Version": sys.version.split()[0],  # Python interpreter version
        }
        return info
    except Exception as e:
        print(f"Error retrieving OS information: {e}")
        return None


<span style="font-family: 'Time New Roman '; font-size: 20px;">
The following cell defines a Tool called search_web(query), <br> it takes one argument and returns either a summary of the response from the query or tells us that no summary was returned. <br>
An example of a successful query is 'Python programming language' or 'BBC'.
</span> 

In [ ]:
import requests

def search_web(query):
    print(f"Called the search_web function with the query {query}.")
    """
    Performs a search query using the DuckDuckGo Instant Answer API 
    and returns a summary text if available.
    """
    url = "https://api.duckduckgo.com/"
    params = {
        "q": query,
        "format": "json",
        "no_html": "1",
        "skip_disambig": "1"
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status() # Raises an error for bad status codes
        data = response.json()
        
        # Look for an abstract summary or related topics
        abstract = data.get("AbstractText")
        if abstract:
            return abstract
        
        # Fallback to the first related topic if no direct abstract exists
        related = data.get("RelatedTopics", [])
        if related and "Text" in related[0]:
            return related[0]["Text"]
            
        return "No direct summary found for this query."
        
    except requests.exceptions.RequestException as e:
        return f"An error occurred: {e}"

# Example usage:
# answer = search_web("Python programming language")
# print(answer)

<span style="font-family: 'Time New Roman '; font-size: 20px;">
The following cell executes the search_web() function directly, <br> simply to see whether a result is forthcoming.
</span> 

In [ ]:
answer = search_web("Python programming language")
print(answer)

In [ ]:
import sys
print(sys.executable)

<span style="font-family: 'Time New Roman '; font-size: 20px;">
The following cell defines the get_share_price() function<br>
it takes one argument - the ticker symbol of a company share.<br>
I found that I needed to install the yfinance package to the Python version that I was running<br>
which was version 3.12.0, I therefore changed the Kernel version that I use for this code.<br>
The Share price that is found seems to be that found on the New York Stock Exchange<br>
If you want the London Stock Exchange you can try appending '.L' to the ticker symbol to see whether there is a match.

</span> 

In [ ]:
import yfinance as yf


def get_share_price(ticker_symbol):
  print(f"Fetching share price for {ticker_symbol}...")
  try:
    # Create a ticker object (e.g., "PRU.L" for London Stock Exchange)
    stock = yf.Ticker(ticker_symbol)

    # Get the latest daily history or current price data
    todays_data = stock.history(period="1d")

    if not todays_data.empty:
      # Extract the closing or current price from the dataframe
      current_price = todays_data["Close"].iloc[-1]
      return f"The latest price for {ticker_symbol} is {current_price:.2f}"
    else:
      return "No price data found for this ticker."

  except Exception as e:
    return f"An error occurred: {e}"


# Example usage (PRU on the London Stock Exchange uses the .L suffix)
print(get_share_price("PRU.L"))

<span style="font-family: 'Time New Roman '; font-size: 20px;">
The following cell executes the get_share_price() function directly,<br>
simply to see whether a result is forthcoming.
</span> 

In [ ]:
answer = get_share_price("PRU")
print(answer)

<span style="font-family: 'Time New Roman '; font-size: 20px;">
The following cell is another example of a direct call to the get_os_info() function<br>
which produces a formatted result.
</span> 

In [ ]:
if __name__ == "__main__":
    os_info = get_os_info()
    if os_info:
        print("=== Operating System Information ===")
        for key, value in os_info.items():
            print(f"{key}: {value}")

In [ ]:
# Additional Imports

import json
from openai import OpenAI
import gradio as gr

<span style="font-family: 'Time New Roman '; font-size: 20px;">
The following cell specifies that we are using Ollama <br> and the model that I downloaded and used is gpt-oss:latest
</span> 

In [ ]:
# Select LLM to use

MODEL = "gpt-oss:latest"
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

<span style="font-family: 'Time New Roman '; font-size: 20px;">
The following cell creates a Global variable containing the system_message.
</span> 

In [ ]:
# Global definition of the LLM's system message.

system_message = """
You are a helpful assistant.
Always be accurate. If you don't know the answer, say so.
"""

<span style="font-family: 'Time New Roman '; font-size: 20px;">
The following cell provides the required definition that identifies <br> the three tools that we are making available to the LLM.
</span> 

In [ ]:
availableTools = [
    {
        "type": "function",
        "function": {
            "name": "get_os_info",
            "description": "Retrieves operating system details, platform release, architecture, processor, and the current Python interpreter version.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "Search the internet.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The query that the user wants to ask the internet.",
                    }
                },
                "required": ["query"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_share_price",
            "description": "Obtain Share Prices.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker_symbol": {
                        "type": "string",
                        "description": "The ticker symbol of the Share for which the user wants the Share Price.",
                    }
                },
                "required": ["ticker_symbol"],
                "additionalProperties": False
            }
        }
    }
]

<span style="font-family: 'Time New Roman '; font-size: 20px;">
The following cell provides the required definition pointing to the definition <br> that we have made referencing the tools that we are making available.
</span> 

In [ ]:
tools = availableTools

<span style="font-family: 'Time New Roman '; font-size: 20px;">
The following cell defines the function that handles the result of the call to the tool.
</span> 

In [ ]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_os_info":
        arguments = json.loads(tool_call.function.arguments)
        os_details = get_os_info()
        response = {
            "role": "tool",
            "content": json.dumps(os_details),
            "tool_call_id": tool_call.id
        }
    if tool_call.function.name == "search_web":
        arguments = json.loads(tool_call.function.arguments)
        question = arguments.get('query')
        web_details = search_web(question)
        response = {
            "role": "tool",
            "content": json.dumps(web_details),
            "tool_call_id": tool_call.id
        }      
    if tool_call.function.name == "get_share_price":
        arguments = json.loads(tool_call.function.arguments)
        question = arguments.get('ticker_symbol')
        share_price = get_share_price(question)
        response = {
            "role": "tool",
            "content": json.dumps(share_price),
            "tool_call_id": tool_call.id
        }              
    return response

<span style="font-family: 'Time New Roman '; font-size: 20px;">
The following cell defines the function that performs the 'chat' operation with gradio as the UI.
</span> 

In [ ]:
def chat(message, history):
    # Format Gradio history safely
    formatted_history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + formatted_history + [{"role": "user", "content": message}]
    
    while True:
        # Get the response from the model
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        choice = response.choices[0]
        response_message = choice.message

        # Check if the model wants to call one or more tools
        if choice.finish_reason == "tool_calls" or getattr(response_message, "tool_calls", None):
            # 1. Append the assistant's tool-call request to messages
            messages.append(response_message)

            # 2. Execute the tool(s) and append the response(s)
            # Note: If your model supports parallel tool calls, handle_tool_call 
            # might need to loop through multiple tool_calls in response_message.tool_calls
            tool_response_message = handle_tool_call(response_message)
            messages.append(tool_response_message)
            
            # Loop continues here, sending the new tool results back to the LLM!
        else:
            # If finish_reason is 'stop' (or anything else), the model is done 
            # and has provided its final text response.
            return response_message.content

<span style="font-family: 'Time New Roman '; font-size: 20px;">
The following cell launchs gradio to provide the UI within which we can interact with the LLM.<br>
I had success with the chat:<br>
Hello, could you list any tools that are available to you in order to perform tool calls, please.<br>
This resulted in the LLM providing a table displaying all three tools, their purpose, parameters and return output.<br>
I then asked:<br>
Could you provide the output from calling the get_os_info tool, then the search_web tool with the query 'BBC' and then could you use the get_share_price tool with the ticker_symbol 'NVDA'  and give me the share price that is returned, please.<br>
The LLM then successfully returned the appropriate output from each tool call in a table and asked if there were anything else that I might like.
</span> 

In [ ]:
gr.ChatInterface(fn=chat).launch()